In [ ]:
import sys
sys.path.append('../src')
from circuit_postprocess import *
from should_be_stdlib import *
from neurodata import *
from circuits import *
from data import *

In [ ]:
from itertools import combinations_with_replacement

In [ ]:
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
from matplotlib import gridspec
import numpy as np
import seaborn as sns

# data loading

In [ ]:
neurons = len(get_tc(load_set()))

# Cross-correlation btrw multiple matrices

In [ ]:
# Create sample matrices
matrices = {
    # name: mirror_matrix(mat.to_numpy())
    # for (name, mat) in {
        'PearsonR': pd.read_csv(datapath('results_correlation-pearson.csv'), index_col=0),
        'Fidelity': pd.read_csv(datapath('results_classical-fidelity.csv'), index_col=0),
        'Euclidean': pd.read_csv(datapath('results_euclidean.csv'), index_col=0),
        # 'Euclidean iFFT': pd.read_csv(datapath('results_euclidean-ifft.csv'), index_col=0), # basically the same
        'Ang': pd.read_csv(datapath('results_simulator_ang.csv'), index_col=0),
        'AngQFT': pd.read_csv(datapath('results_simulator_ang-qft.csv'), index_col=0),
        'Amp': pd.read_csv(datapath('results_simulator_amp.csv'), index_col=0),
        'AmpQFT': pd.read_csv(datapath('results_simulator_amp-qft.csv'), index_col=0),
        # 'Ang (IBM)': qpu_ang,
        # 'Amp (IBM)': qpu_amp,
        # 'Amp +DDD (IBM)': qpu_amd_ddd,
        # 'AmpQFT (IBM)': qpu_amp_qft,
        # 'AmpQFT +DDD (IBM)': qpu_amp_qft_ddd,
    # }.items()
}
N = len(matrices)

In [ ]:
corrs = {
    (i, j):
    (
        namei, namej,
        np.corrcoef(
            mirror_matrix(mati.to_numpy()),
            mirror_matrix(matj.to_numpy()),
        )
    )
    for (
        (i, (namei, mati)),
        (j, (namej, matj))
    ) in combinations_with_replacement(list(enumerate(matrices.items())), 2)
}

In [ ]:
fig = plt.figure(figsize=(N * 2, N * 2))
gs = gridspec.GridSpec(N, N, wspace=0.2, hspace=0.2)

# Plot original heatmaps on the main diagonal and correlation heatmaps
for ((i, j), (namei, namej, corr)) in corrs.items():
    # if i == j:
    #     # Plot original heatmaps on the main diagonal
    #     ax = fig.add_subplot(gs[i, j])
    #     upper_triangle_matrix = matrices[i]
    #     # upper_triangle_matrix = replace_bottom_triangle_with_nan(upper_triangle_matrix)  # to eliminate lower triangle
    #     sns.heatmap(upper_triangle_matrix, ax=ax, cbar=False, square=True, cmap='magma')  # , vmin=0, vmax=1)

    # elif i < j:

    # Plot correlation heatmaps in the upper triangle
    ax = fig.add_subplot(gs[i, j])
    sns.heatmap(
        corr[neurons:, :neurons],
        ax = ax,
        cbar = False,
        square = True,
        cmap = 'magma'
    )  # , vmin=0, vmax=1)

    # clear labels
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')

    if i == 0:
        ax.set_title(namej, fontsize=12)  # Set title for top row
    if i == j:
        ax.set_ylabel(namei, fontsize=12, rotation=0, ha='right')  # Set ylabel for left column

fig.suptitle('Cross-Correlation', fontsize=24)
fig.subplots_adjust(top=0.93)  # move suptitle down
# plt.tight_layout()

plt.savefig(figspath('analysis_cross-correlation.png'), dpi=300)
plt.show() # big figure, dont show it
plt.clf()
plt.close()

## heatmaps of main diagonal specifically

Just the main diagonal into 4 different plots for closer viewing


In [ ]:
plt.rcParams['font.size'] = 16
fig, axs = plt.subplots(1, 3, figsize=(10, 4))
for (ax, name) in zip(axs.flatten(), ['PearsonR','Fidelity','Euclidean']):
    sns.heatmap(mirror_matrix(matrices[name].to_numpy()), ax=ax, cmap='magma', cbar=False, square=True)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(name)
plt.suptitle('Classical Metrics', fontsize=20)
plt.tight_layout(pad=1)
plt.savefig(figspath('metrics_classical.png'), dpi=300)
plt.show()
plt.close()

In [ ]:
plt.rcParams['font.size'] = 16
fig, axs = plt.subplots(1, 3, figsize=(10, 4))
for (ax, name) in zip(axs.flatten(), ['Ang','Amp','AmpQFT']):
    sns.heatmap(mirror_matrix(matrices[name].to_numpy()), ax=ax, cmap='magma', cbar=False, square=True)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(name)
plt.suptitle('Quantum Metrics', fontsize=20)
plt.tight_layout(pad=1)
plt.savefig(figspath('metrics_circuits.png'), dpi=300)
plt.show()
plt.close()

# statistical testing
- the proper test for 2 distance matrices is actually Mantel's test.
- we can use this if we have *distance* matrices to measure the correlation (-1 to +1)
- https://fukamilab.github.io/BIO202/06-C-matrix-comparison.html
- says spearman method should be use when non-linearity is expected, which we use here
- as implemented in skbio, mantel's test requires the diagonals to be 0
- this means that we should invert state fidelity (1-fidelity) bc the middle diagonal is 1, not 0
- The diagonal for state fidelity is not exactly 0, but it's very close to 0 so we can use Mantel's test anyways

In [ ]:
from skbio.stats.distance import mantel, pwmantel

mantel_result = pwmantel(
    [hollow_matrix(mirror_matrix(m.to_numpy())) for m in matrices.values()],
    labels = matrices.keys(),
    method = 'spearman',
    seed = 0,
)[['statistic', 'p-value']]

# rank correlation shows how correlated the ranks of data are
mantel_result.to_csv(datapath('analysis_mantel.csv'))
mantel_result_grid = mantel_result.pivot_table(index='dm1', columns='dm2')
mantel_result

In [ ]:
mantel_result_statistic = \
mantel_result_grid['statistic'].loc[
    mantel_result_grid['statistic'].isna().sum(axis=1).sort_values(ascending=True).index,
    mantel_result_grid['statistic'].isna().sum(axis=0).sort_values(ascending=False).index,
]

mantel_result_statistic.to_csv(datapath('analysis_mantel_statistic.csv'))

In [ ]:
mantel_result_pvalue = \
mantel_result_grid['p-value'].loc[
    mantel_result_grid['p-value'].isna().sum(axis=1).sort_values(ascending=True).index,
    mantel_result_grid['p-value'].isna().sum(axis=0).sort_values(ascending=False).index,
]

mantel_result_pvalue.to_csv(datapath('analysis_mantel_pvalue.csv'))

In [ ]:
plt.rcParams['font.size'] = 16
fig, ax = plt.subplots(figsize=[x*2 for x in mantel_result_statistic.shape])

sns.heatmap(np.abs(mantel_result_statistic), annot=True, cmap='magma_r', cbar=False, fmt='.3f')
plt.title('Mantel Test Statistic (Absolute Value)', fontsize=20)
plt.xlabel(None)
plt.ylabel(None)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(figspath('analysis_mantel_statistic.png'), dpi=300)
plt.show()
plt.close()

In [ ]:
plt.rcParams['font.size'] = 16
fig, ax = plt.subplots(figsize=[x*2 for x in mantel_result_pvalue.shape])

sns.heatmap(mantel_result_pvalue, annot=True, cmap='magma_r', cbar=False, fmt='.3f')
plt.plot(size=(12, 10))
plt.title('Mantel Test (p-Value)', fontsize=20)
plt.xlabel(None)
plt.ylabel(None)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(figspath('analysis_mantel_pvalue.png'), dpi=300)
plt.show()
plt.close()